# Data Loading & EDA
**Datasets:** covid-vax-stance (Poddar et al., ICWSM 2022) and TweetEval stance_abortion (SemEval-2016)


**Note**: All the results will be saved and read on/from the local Google Drive in `StanceProject` directory.

In [ ]:
!pip install -q pandas matplotlib seaborn wordcloud scikit-learn datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/StanceProject'
for d in ['data/raw', 'data/processed', 'plots', 'results', 'predictions', 'models']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from datasets import load_dataset
import re, warnings, urllib.request
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## covid-vax-stance

In [ ]:
base = 'https://raw.githubusercontent.com/sohampoddar26/covid-vax-stance/main/dataset/'
files = ['our_labelled_data.csv', 'our_labelled_data_additional.csv', 'cotfas_data.csv']

frames = []
for fname in files:
    dest = f'{PROJECT}/data/raw/{fname}'
    if not os.path.exists(dest):
        urllib.request.urlretrieve(base + fname, dest)
    frames.append(pd.read_csv(dest))

cvs_raw = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(cvs_raw)} rows | columns: {cvs_raw.columns.tolist()}')

In [ ]:
CVS_MAP = {'ProVax': 'FAVOR', 'AntiVax': 'AGAINST', 'Neutral': 'NONE'}

cvs = cvs_raw[['tweet', 'majority_label']].rename(columns={'tweet': 'text', 'majority_label': 'label'})
cvs = cvs.dropna(subset=['text', 'label'])
cvs['dataset'] = 'covid-vax-stance'
cvs['label_unified'] = cvs['label'].map(CVS_MAP).fillna(cvs['label'])

print('Labels:', cvs['label'].value_counts().to_dict())
print('Total: ', len(cvs))

## TweetEval stance_abortion

In [ ]:
te_raw = load_dataset('cardiffnlp/tweet_eval', 'stance_abortion')
lnames = te_raw['train'].features['label'].names
print('Label names:', lnames)

TE_MAP = {'favor': 'FAVOR', 'against': 'AGAINST', 'none': 'NONE'}

parts = []
for sname, split in te_raw.items():
    d = pd.DataFrame({'text': split['text'], 'label_id': split['label']})
    d['label'] = d['label_id'].map(lambda i: lnames[i])
    d['split'] = sname
    parts.append(d)

te = pd.concat(parts, ignore_index=True)
te['dataset'] = 'tweeteval-stance'
te['label_unified'] = te['label'].map(TE_MAP).fillna(te['label'])

print('Labels:', te['label'].value_counts().to_dict())
print('Total: ', len(te))

## Save processed data

In [ ]:
cvs.to_csv(f'{PROJECT}/data/processed/covid_vax_stance.csv', index=False)
te.to_csv( f'{PROJECT}/data/processed/tweeteval_stance.csv', index=False)

print(f'{PROJECT}/data/processed/covid_vax_stance.csv ({len(cvs)} rows)')
print(f'{PROJECT}/data/processed/tweeteval_stance.csv ({len(te)} rows)')

## Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, df, title in [(axes[0], cvs, 'covid-vax-stance'), (axes[1], te, 'TweetEval stance_abortion'),]:
    counts = df['label'].value_counts()
    bars = ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJECT}/plots/label_distribution.png', bbox_inches='tight')
plt.show()

## Text length distribution

In [ ]:
cvs['text_len'] = cvs['text'].str.len()
te['text_len']  = te['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(cvs['text_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('covid-vax-stance - tweet length')
axes[1].hist(te['text_len'],  bins=40, color='coral', edgecolor='white')
axes[1].set_title('TweetEval - tweet length')
for ax in axes:
    ax.set_xlabel('Characters')
plt.tight_layout()
plt.savefig(f'{PROJECT}/plots/text_length.png', bbox_inches='tight')
plt.show()

for name, df in [('covid-vax-stance', cvs), ('TweetEval', te)]:
    print(f'{name}: mean={df["text_len"].mean():.0f}, median={df["text_len"].median():.0f}, max={df["text_len"].max()}')

## Word clouds per label

In [ ]:
def clean_for_wc(text):
    text = re.sub(r'http\S+', '', str(text))
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.lower().strip()

def plot_wordclouds(df, name):
    labels = sorted(df['label_unified'].dropna().unique())
    fig, axes = plt.subplots(1, len(labels), figsize=(5 * len(labels), 4))
    for ax, lbl in zip(axes, labels):
        corpus = ' '.join(df[df['label_unified'] == lbl]['text'].apply(clean_for_wc))
        wc = WordCloud(width=400, height=300, background_color='white', max_words=80).generate(corpus or 'empty')
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'{lbl}', fontsize=12)
    fig.suptitle(name, fontsize=13, fontweight='bold')
    plt.tight_layout()
    safe = name.lower().replace(' ', '_').replace('-', '_')
    plt.savefig(f'{PROJECT}/plots/wordcloud_{safe}.png', bbox_inches='tight')
    plt.show()

plot_wordclouds(cvs, 'covid-vax-stance')
plot_wordclouds(te, 'TweetEval stance_abortion')

## Dataset summary

In [ ]:
summary = []
for name, df in [('covid-vax-stance', cvs), ('TweetEval stance_abortion', te)]:
    row = {'dataset': name, 'total': len(df), 'avg_len': round(df['text_len'].mean(), 1)}
    for lbl, cnt in df['label_unified'].value_counts().items():
        row[lbl] = cnt
    summary.append(row)

summary_df = pd.DataFrame(summary).set_index('dataset')
summary_df.to_csv(f'{PROJECT}/results/dataset_summary.csv')
print(summary_df.to_string())